# 🛠️ Taranga Notebook Updater Utility
This utility provides a programmatic way to inject metrics and documentation into Jupyter Notebooks. It ensures that standard diagnostic outputs like Classification Reports and Confusion Matrices are present in technical reports.

In [ ]:
import json
import os

def add_metrics_to_notebook(notebook_path):
    """Injects standard classification metrics and confusion matrices into a target notebook."""
    if not os.path.exists(notebook_path):
        print(f"Error: {notebook_path} not found.")
        return

    with open(notebook_path, 'r') as f:
        nb = json.load(f)

    # 1. Standard Classification Report Cell
    report_cell = {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Standard Text-Based Classification Report\n",
            "print(\"### Standard Classification Report ###\\n\")\n",
            "print(classification_report(y_test, rf.predict(X_test), target_names=LABEL_COLS))"
        ]
    }

    # 2. Confusion Matrices Section
    cm_section_md = {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### 3.3 Binary Confusion Matrices\n",
            "We use binary confusion matrices to verify the precise True Positive, True Negative, False Positive, and False Negative counts for each specific LD domain."
        ]
    }

    # 3. Confusion Matrix Plotting Cell
    cm_plotting_cell = {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "from sklearn.metrics import multilabel_confusion_matrix\n",
            "\n",
            "mcm = multilabel_confusion_matrix(y_test, rf.predict(X_test))\n",
            "\n",
            "plt.figure(figsize=(18, 10))\n",
            "for i, ld in enumerate(LABEL_COLS):\n",
            "    plt.subplot(2, 3, i+1)\n",
            "    sns.heatmap(mcm[i], annot=True, fmt='d', cmap='Blues', cbar=False, \n",
            "                xticklabels=['Pred Neg', 'Pred Pos'], \n",
            "                yticklabels=['Actual Neg', 'Actual Pos'])\n",
            "    plt.title(f'Confusion Matrix: {ld.upper()}', fontsize=12, fontweight='bold')\n",
            "\n",
            "plt.suptitle('Binary Confusion Matrices per LD Domain', fontsize=16, fontweight='bold')\n",
            "plt.tight_layout(rect=[0, 0.03, 1, 0.95])\n",
            "plt.show()"
        ]
    }

    # Find insertion point (search for performance map heatmap)
    idx = -1
    for i, cell in enumerate(nb['cells']):
        if cell['cell_type'] == 'code' and any("Classification Performance Map" in s for s in cell.get('source', [])):
            idx = i
            break

    if idx != -1:
        # Check if already updated to prevent duplicates
        if any("multilabel_confusion_matrix" in "".join(c.get('source', [])) for c in nb['cells']):
             print(f"Notebook {notebook_path} is already updated.")
             return
             
        nb['cells'].insert(idx + 1, cm_plotting_cell)
        nb['cells'].insert(idx + 1, cm_section_md)
        nb['cells'].insert(idx + 1, report_cell)
        
        with open(notebook_path, 'w') as f:
            json.dump(nb, f, indent=1)
        print(f"✅ Successfully updated {notebook_path}")
    else:
        print(f"❌ Could not find insertion point in {notebook_path}")

## Execution
Run the cell below to apply the updates to the standard training notebook.

In [ ]:
target_nb = '../notebooks/02_model_trainer.ipynb'
add_metrics_to_notebook(target_nb)